In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
import xgboost as xgb
import joblib
import seaborn as sns

# Paths to datasets
# Paths to the dataset
train_dir = "/kaggle/input/rgb-skin/RGB/train"
valid_dir = "/kaggle/input/rgb-skin/RGB/valid"
test_dir = "/kaggle/input/rgb-skin/RGB/test"

num_classes = 4

# Hyperparameters
image_size = (299, 299)
batch_size = 32

# Data Augmentation and Loading
datagen = ImageDataGenerator(rescale=1.0/255.0)
train_gen = datagen.flow_from_directory(
    train_dir, target_size=image_size, batch_size=batch_size, class_mode='categorical', shuffle=True)
valid_gen = datagen.flow_from_directory(
    valid_dir, target_size=image_size, batch_size=batch_size, class_mode='categorical', shuffle=False)
test_gen = datagen.flow_from_directory(
    test_dir, target_size=image_size, batch_size=batch_size, class_mode='categorical', shuffle=False)

# Load InceptionV3 as feature extractor
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(*image_size, 3))
x = GlobalAveragePooling2D()(base_model.output)
feature_extractor = Model(inputs=base_model.input, outputs=x)

# Extract features and labels
def extract_features_and_labels(generator):
    features, labels = [], []
    for i, (images, label_batch) in enumerate(generator):
        features_batch = feature_extractor.predict(images)
        features.extend(features_batch)
        labels.extend(label_batch)
        if i >= len(generator) - 1:
            break
    return np.array(features), np.argmax(labels, axis=1)

train_features, train_labels = extract_features_and_labels(train_gen)
valid_features, valid_labels = extract_features_and_labels(valid_gen)
test_features, test_labels = extract_features_and_labels(test_gen)

# Train XGBoost model
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=num_classes,
    eval_metric='mlogloss',
    use_label_encoder=False
)
xgb_model.fit(train_features, train_labels, eval_set=[(valid_features, valid_labels)], early_stopping_rounds=10, verbose=True)

# Evaluate the model
y_pred = xgb_model.predict(test_features)
print("Classification Report:")
print(classification_report(test_labels, y_pred, target_names=test_gen.class_indices.keys()))

# Confusion Matrix
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_gen.class_indices.keys(), yticklabels=test_gen.class_indices.keys())
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Save the best model
joblib.dump(xgb_model, "best_skin_cancer_model.pkl")
print("Model saved as best_skin_cancer_model.pkl")

# Plotting loss and accuracy (Inception Training Phase)
# Assuming validation loss/accuracy during feature extraction
history = feature_extractor.history  # Replace with training history if training Inception was done
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.show()


Found 3028 images belonging to 4 classes.
Found 280 images belonging to 4 classes.
Found 88 images belonging to 4 classes.
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


I0000 00:00:1733113997.655032     150 service.cc:145] XLA service 0x7cb66c003a20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1733113997.655090     150 service.cc:153]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0


1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step


I0000 00:00:1733114005.556568     150 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━

KeyError: 'loss'